In [10]:
%pip install optuna optuna-dashboard

import pandas as pd
import numpy as np
import torch
import optuna
from optuna.pruners import MedianPruner, PatientPruner
from torch.optim.lr_scheduler import ReduceLROnPlateau
import torch.optim as optim
from pathlib import Path
import time
import cloud_sync
import dataclasses

import datasets as ds
import params as params_module
import model_factory
from convCNP.training.training_elev import (
    get_fold_holdout_indices,
    train_epoch_smacnp,
    eval_epoch_smacnp,
)
from convCNP.training.utils import get_fold_data_smacnp



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [11]:
# ── Constants ─────────────────────────────────────────────────────────────
PW_FIRST_DATE         = '2017-01-01'
PW_LAST_DATE          = '2024-12-31'
PW_VALIDITY_THRESHOLD = 0.99
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Base params (defined first so data paths are available) ───────────────
base_params = params_module.Params(VARIABLE='tmax', MODEL_TYPE='smacnp',
                                   CONTEXT_TYPE='peakweather',
                                   RUN_TYPE='cloud' if params_module.is_renku() else 'local')
base_params = base_params.with_data_paths(ds.build_data_paths(Path('datasets')))

# ── Sync datasets from remote storage (needed on Renku) ───────────────────
BASE_DATASET_DIR  = Path('datasets')
REMOTE_DATASET_DIR = Path().cwd() / '..' / 'datasets-mount'
cloud_sync.sync_from_cloud_if_needed(REMOTE_DATASET_DIR, BASE_DATASET_DIR, base_params.RUN_TYPE)

# ── Load data (once, reused across all trials) ────────────────────────────
daily_tmean, stations_meta, valid_stations = ds.load_peakweather_stations(
    first_date=PW_FIRST_DATE, last_date=PW_LAST_DATE,
    validity_threshold=PW_VALIDITY_THRESHOLD, aggregation_method='mean',
)

pw_dates_pd  = pd.DatetimeIndex(daily_tmean.index.date.astype(str))
pw_times_np  = np.array(pw_dates_pd, dtype='datetime64[D]')
seasonal_features = ds.compute_seasonal_features(pw_times_np, device=DEVICE)

hi_res_elevation, hi_res_tpi = ds.load_high_res_topography(
    base_params.HI_RES_TOPOGRAPHY_ZARR_PATH
)

train_stations, _, _ = ds.split_peakweather_stations(
    valid_stations, train_frac=0.8, val_frac=0.0, seed=42
)

pw_metadata = ds.compute_pw_metadata(daily_tmean, train_stations)

x_pool, y_pool_raw = ds.build_pw_station_tensors(
    daily_tmax=daily_tmean, station_ids=train_stations,
    stations_meta=stations_meta, hi_res_tpi=hi_res_tpi,
    metadata=pw_metadata, seasonal_features=seasonal_features,
    dates_pd=pw_dates_pd, device=DEVICE,
)
y_pool_raw     = y_pool_raw.unsqueeze(-1)
y_context_pool = torch.nan_to_num(y_pool_raw, nan=0.0)
y_target_pool  = y_pool_raw.squeeze(-1)

print(f"x_pool: {x_pool.shape}  |  device: {DEVICE}")

# ── Objective function ────────────────────────────────────────────────────
def objective(trial: optuna.Trial) -> float:

    # Sample hyperparameters
    r_dim      = trial.suggest_categorical('r_dim',       [64, 128, 256, 512])
    v_dim      = trial.suggest_categorical('v_dim',       [64, 128, 256, 512])
    w_dim      = trial.suggest_categorical('w_dim',       [64, 128, 256, 512])
    hidden     = trial.suggest_categorical('hidden',      [256, 512, 1024])
    n_heads    = trial.suggest_categorical('n_heads',     [4, 8, 16])
    mlp_layers = trial.suggest_categorical('mlp_layers',  [2, 3])
    dropout    = trial.suggest_float('dropout', 0.0, 0.3, step=0.05)
    laplace_p  = trial.suggest_categorical('laplace_p',  [1.0, 2.0])
    lr         = trial.suggest_float('lr', 1e-4, 5e-4, log=True)
    pe_dim     = trial.suggest_categorical('pe_dim',      [32, 64])
    layer_norm = trial.suggest_categorical('layer_norm',  [True, False])

    # Enforce divisibility constraint
    if r_dim % n_heads != 0 or v_dim % n_heads != 0:
        raise optuna.TrialPruned()

    # Build params for this trial
    trial_params = dataclasses.replace(
        base_params,
        R_DIM=r_dim, V_DIM=v_dim, W_DIM=w_dim,
        SMACNP_HIDDEN=hidden, N_HEADS=n_heads,
        NUM_LAYERS_MLP=mlp_layers, PE_DIM=pe_dim,
        DROPOUT_RATE=dropout,
        LAPLACE_P=laplace_p,
        USE_LAYER_NORM=layer_norm,
        LOSS_FN='gll',
    )

    # Build model
    model, loss_fn, get_value_fn = model_factory.build_smacnp(
        trial_params, x_pool.shape[-1]
    )
    model = model.to(DEVICE)

    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5,
                                  patience=20, min_lr=1e-6)

    # Training settings
    FOLD           = 0
    N_FOLDS        = 5
    MAX_EPOCHS     = 1500
    WARMUP         = 150
    CHECK_INTERVAL = 10

    n_samples        = x_pool.shape[0]
    start, end       = get_fold_holdout_indices(FOLD, N_FOLDS, n_samples)
    best_val_loss    = float('inf')

    for epoch in range(MAX_EPOCHS):
        training_data, held_out = get_fold_data_smacnp(
            (start, end), x_pool, y_context_pool, x_pool, y_target_pool,
            batch_size=16, shuffle=True,
        )

        train_epoch_smacnp(
            model, optimizer, training_data, loss_fn,
            device=DEVICE,
            context_fraction=(0.2, 0.8),
            exclusive_context=False,
        )

        val_loss, _, _, _ = eval_epoch_smacnp(
            model, held_out, loss_fn, get_value_fn,
            device=DEVICE,
            exclusive_context=False,
        )

        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss

        # Report to Optuna for pruning (after warmup, every CHECK_INTERVAL epochs)
        if epoch >= WARMUP and epoch % CHECK_INTERVAL == 0:
            trial.report(val_loss, epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()

    return best_val_loss


# ── Progress callback ─────────────────────────────────────────────────────
hpo_start = time.time()

def progress_callback(study: optuna.Study, trial: optuna.Trial):
    elapsed    = time.time() - hpo_start
    n_complete = len([t for t in study.trials
                      if t.state == optuna.trial.TrialState.COMPLETE])
    n_pruned   = len([t for t in study.trials
                      if t.state == optuna.trial.TrialState.PRUNED])
    n_done     = n_complete + n_pruned
    n_remaining = 50 - len(study.trials)   # adjust 50 to n_trials below

    avg_s = elapsed / max(n_done, 1)
    eta_s = avg_s * max(n_remaining, 0)

    val = f"{trial.value:.4f}" if trial.value is not None else "pruned"
    print(
        f"Trial {trial.number:3d} | val={val} | "
        f"best={study.best_value:.4f} | "
        f"complete={n_complete} pruned={n_pruned} | "
        f"elapsed={elapsed/60:.1f}min | ETA={eta_s/60:.1f}min"
    )


# ── Run the study ─────────────────────────────────────────────────────────
pruner = PatientPruner(MedianPruner(n_startup_trials=20), patience=5)

study = optuna.create_study(
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=pruner,
    storage='sqlite:///hpo_smacnp_pw.db',
    study_name='smacnp_pw_hpo',
    load_if_exists=True,
)

study.optimize(objective, n_trials=50, show_progress_bar=True,
               callbacks=[progress_callback])

print(f"\nBest trial:  val_loss={study.best_value:.4f}")
print(f"Best params: {study.best_trial.params}")


Cloud run detected. Syncing data from /home/renku/work/convNPClimate/../datasets-mount to datasets...
Synchronizing directory datasets with /home/renku/work/convNPClimate/../datasets-mount
Source directory: /home/renku/work/convNPClimate/../datasets-mount:


dirsync finished in 1.80 seconds.
28 directories parsed, 0 files copied

Dataset sync complete.
Loading PeakWeather stations (None) from 2017-01-01 to 2024-12-31 ...
  172 / 302 stations pass validity threshold (99%)
  Daily tmax shape: (2921, 172)  (2017-01-01 → 2024-12-30)
  Altitude range: [203, 3571] m
Computed seasonal features: torch.Size([2921, 2]), dtype: torch.float32
Loading high-resolution topography from Zarr: datasets/topo_subset.zarr
Topography dataset dimensions: FrozenMappingWarningOnValuesAccess({'y': 12800, 'x': 14200})
DEM shape: (12800, 14200), dtype: int16
TPI (TPI_500M) shape: (12800, 14200), dtype: float32
Station split: 137 train / 0 val / 35 test
PW train stats: mean=280.93 K, std=8.07 K
x_pool: torch.Size([2921, 137, 6])  |  device: cuda


/tmp/ipykernel_6427/4118253761.py:161: ExperimentalWarning: PatientPruner is experimental (supported from v2.8.0). The interface can change in the future.
  pruner = PatientPruner(MedianPruner(n_startup_trials=20), patience=5)
[I 2026-08-27 22:03:19,071] Using an existing study with name 'smacnp_pw_hpo' instead of creating a new one.
  0%|          | 0/50 [00:00<?, ?it/s]

Built SMACNP with 2,526,083 trainable parameters


  0%|          | 0/50 [58:35<?, ?it/s]


[W 2026-08-27 23:01:54,528] Trial 2 failed with parameters: {'r_dim': 128, 'v_dim': 512, 'w_dim': 512, 'hidden': 256, 'n_heads': 16, 'mlp_layers': 2, 'dropout': 0.2, 'laplace_p': 2.0, 'lr': 0.00018033330377234335, 'pe_dim': 64, 'layer_norm': False} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/home/renku/work/.venv/lib/python3.13/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "/tmp/ipykernel_6427/4118253761.py", line 109, in objective
    train_epoch_smacnp(
    ~~~~~~~~~~~~~~~~~~^
        model, optimizer, training_data, loss_fn,
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<2 lines>...
        exclusive_context=False,
        ^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/home/renku/work/convNPClimate/convCNP/training/training_elev.py", line 402, in train_epoch_smacnp
    obj, opt, model = train_batch_smacnp(task, opt, model, ll, device=device,
                   

KeyboardInterrupt: 